通过隐变量存储之前的序列信息，基于当前输入 $x_t$ 和隐状态 $h_{t-1}$ 计算当权输出

**如何评估一个语言模型的质量：困惑度**

采用交叉熵损失 $\mathcal{L} = -\sum_{i=0}^{V-1} y_i \cdot \log(p_i)$

In [1]:
"Load Data"

import re
import collections
from collections import Counter
import math

def read_text_file(file_path: str) -> list[str]:

    # 打开文件：指定utf-8编码，遇异常字符自动替换，避免乱码报错
    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
        lines = f.readlines()

    # 逐行清洗，额外过滤空行
    cleaned_lines = []
    for line in lines:
        # 正则：所有非英文字母的连续字符，替换为单个空格
        line_processed = re.sub('[^A-Za-z]+', ' ', line)
        # 去首尾空格 + 全小写
        line_processed = line_processed.strip().lower()
        # 过滤空行（原d2l未处理，属于优化点）
        if line_processed:
            cleaned_lines.append(line_processed)

    return cleaned_lines

def tokenize(lines, token='word'):
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('unknowed token:' + token)

def count_corpus(tokens):
    "统计语料词频"

    # 空输入直接返回空计数器
    if len(tokens) == 0:
        return Counter()
    # 判断是二维嵌套列表（每行分词结果），展平为一维
    if isinstance(tokens[0], list):
        tokens = [token for line in tokens for token in line]
    return Counter(tokens)


class Vocab:
    
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        # 空输入默认初始化为空列表
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []

        # 1. 统计全部词元频率，并按频率从高到低排序
        counter = count_corpus(tokens)
        # 按词频降序排列 (token, freq)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)

        # 2. 初始化索引-词双向映射
        # 首位固定<unk>未知词，再拼接用户自定义保留词
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}

        # 3. 过滤低频词，构建完整词表映射
        for token, freq in self._token_freqs:
            # 低于最小频率阈值直接跳过，后续全部不再遍历
            if freq < min_freq:
                break
            # 避免和保留token重复
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):

        # 单个词元输入
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        # 列表/元组批量转换
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[idx] for idx in indices]

    @property
    def unk(self):
        "未知词元<unk>固定索引为0"
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

def load_text_data(path, max_tokens=-1):

    lines = read_text_file(path)
    tokens = tokenize(lines, 'word')
    vocab = Vocab(tokens) 
    corpus = [vocab[token] for line in tokens for token in line]

    if max_tokens > 0:
        corpus = corpus[:max_tokens]

    return corpus, vocab

def load_data(batch_size, num_steps):

    file_path = r'C:\Users\17934\Desktop\MachineLearning\Program\TextData.txt'

    corpus, vocab = load_text_data(file_path)
    corpus = torch.tensor(corpus, dtype=torch.long)

    X_raw = corpus[:-1]
    Y_raw = corpus[1:]
    total_len = len(X_raw)

    num_batches = total_len // (batch_size * num_steps)
    valid_len = num_batches * batch_size * num_steps
    X_raw = X_raw[:valid_len].reshape(batch_size, -1)
    Y_raw = Y_raw[:valid_len].reshape(batch_size, -1)

    def data_iter():
        for i in range(0, X_raw.shape[1], num_steps):
            X_batch = X_raw[:, i:i+num_steps]
            Y_batch = Y_raw[:, i:i+num_steps]
            yield X_batch, Y_batch

    return data_iter, vocab

In [2]:
import torch
from torch import nn
from torch.nn import functional as F

batch_size = 32 # 批次大小，即并行独立文本的数量
num_steps = 35 # 单条序列窗口长度

train_iter_fun, vocab = load_data(batch_size, num_steps)


In [3]:
"构建模型"

num_hiddens = 512
rnn_layer = nn.RNN(len(vocab), num_hiddens, num_layers=2)
state = torch.zeros((2, batch_size, num_hiddens)) # (隐藏层数,批量大小,隐藏单元数)

X = torch.rand(size=(num_steps, batch_size, len(vocab)))
Y, new_state = rnn_layer(X, state) # 注意这里的 Y 不涉及输出运算，只是每个时间步的隐状态

# RNN封装模型类
class RNNModel(nn.Module):
    
    def __init__(self, rnn_layer, vocab_size, **kwargs):
        super(RNNModel, self).__init__(**kwargs)
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size

        # 区分单向/双向RNN
        if not self.rnn.bidirectional:
            self.num_directions = 1
            self.linear = nn.Linear(self.num_hiddens, self.vocab_size)
        else:
            self.num_directions = 2
            # 双向RNN拼接前后向隐状态，维度×2
            self.linear = nn.Linear(self.num_hiddens * 2, self.vocab_size)

    def forward(self, inputs, state):
        # inputs shape (batch, num_steps)
        X = F.one_hot(inputs.T.long(), self.vocab_size)
        X = X.to(torch.float32)

        Y, state = self.rnn(X, state)
        output = self.linear(Y.reshape((-1, Y.shape[-1])))

        return output, state

    def begin_state(self, device, batch_size=1):
        # 判断：RNN/GRU隐状态是单张量；LSTM是(h,c)二元组
        if not isinstance(self.rnn, nn.LSTM):
            # RNN/GRU：(directions * layers, batch, hidden)
            return torch.zeros((
                self.num_directions * self.rnn.num_layers,
                batch_size, self.num_hiddens), device=device)
        else:
            # LSTM：返回 (h0, c0) 两个同形状全0张量
            return (
                torch.zeros((self.num_directions * self.rnn.num_layers, batch_size, self.num_hiddens), device=device),
                torch.zeros((self.num_directions * self.rnn.num_layers, batch_size, self.num_hiddens), device=device)
            )

In [4]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

device = get_device()

net = RNNModel(rnn_layer, len(vocab))
net = net.to(device)

In [5]:
def predict_ch8(prefix, num_preds, net, vocab, device):
    """
    在给定前缀 prefix 字符串后，自动生成 num_preds 个新字符
    net: 训练好的RNNModelScratch模型
    vocab: 字符词表
    """
    # 初始化单样本隐状态
    state = net.begin_state(batch_size=1, device=device)
    # outputs存储全部字符索引，先存入第一个前缀字符的编码
    outputs = [vocab[prefix[0]]]
    # 构造输入函数 
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))

    # ========== 预热期 warm-up: 遍历 prefix 剩余字符，只更新隐状态，不生成预测 ==========
    for y_char in prefix[1:]:
        # 前向传播，丢弃输出，只更新记忆state
        _, state = net(get_input(), state)
        # 把当前前缀字符编码存入序列
        outputs.append(vocab[y_char])

    # ========== 预测阶段: 循环生成 num_preds 个新字符 ==========
    for _ in range(num_preds):
        # 送入上一步最后字符，得到本轮预测得分 + 更新后的隐状态
        y_logits, state = net(get_input(), state)
        # argmax 选出概率最大的字符索引，转 int 存入 outputs
        pred_idx = int(y_logits.argmax(dim=1).item())
        outputs.append(pred_idx)

    # 将所有数字索引转回字符，拼接成完整字符串返回
    return ''.join([vocab.idx_to_token[i] for i in outputs])

In [6]:
def train_ch8(model: nn.Module, train_iter_fn, vocab, lr: float, num_epochs: int, device: torch.device):
    """
    Args:
        model: RNN 模型
        train_iter_fn: 数据迭代器生成函数
    """
    # ---------- 训练组件初始化 ----------
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(reduction="mean")
    model.to(device)

    # 全局性能统计
    total_tokens = 0

    for epoch in range(num_epochs):
        model.train()
        epoch_total_loss = 0.0
        epoch_token_count = 0
        train_iter = train_iter_fn()  # 每轮生成全新迭代器，支持多轮训练

        # 初始化首轮隐状态
        first_X = next(iter(train_iter))[0]
        state = model.begin_state(device=device, batch_size=first_X.shape[0])

        for X, Y in train_iter:
            # 数据迁移到目标设备，保证与模型参数同设备
            X = X.to(device)
            Y = Y.to(device)

            # 关键：隐状态从计算图中剥离，避免反向传播跨批次累积
            if isinstance(state, tuple):
                state = tuple(s.detach() for s in state)
            else:
                state = state.detach()

            # 前向传播
            y_hat, state = model(X, state)

            # 标签维度对齐：(batch, num_steps)转置后展平，匹配输出顺序
            loss = loss_fn(y_hat, Y.T.reshape(-1))

            # 反向传播 + 梯度裁剪 + 参数更新
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # RNN标配，防止梯度爆炸
            optimizer.step()

            # 指标统计
            batch_tokens = Y.numel()
            epoch_total_loss += loss.item() * batch_tokens
            epoch_token_count += batch_tokens
            total_tokens += batch_tokens

        # ---------- 每轮日志输出 ----------
        avg_loss = epoch_total_loss / epoch_token_count
        perplexity = math.exp(avg_loss)

        # 控制日志频率：首末轮+每10轮输出
        if epoch == 0 or (epoch + 1) % 10 == 0 or epoch == num_epochs - 1:
            print(f"Epoch [{epoch+1:>3d}/{num_epochs}] | "
                  f"Perplexity: {perplexity:>6.2f}")


In [9]:
num_epochs = 100
lr = 0.8

train_ch8(net, train_iter_fun, vocab, lr, num_epochs, device)

Epoch [  1/100] | Perplexity:  10.52
Epoch [ 10/100] | Perplexity:   8.19
Epoch [ 20/100] | Perplexity:   6.30
Epoch [ 30/100] | Perplexity:   4.98
Epoch [ 40/100] | Perplexity:   4.08
Epoch [ 50/100] | Perplexity:   3.37
Epoch [ 60/100] | Perplexity:   2.90
Epoch [ 70/100] | Perplexity:   2.50
Epoch [ 80/100] | Perplexity:   2.19
Epoch [ 90/100] | Perplexity:   1.99
Epoch [100/100] | Perplexity:   1.79


In [12]:
predict_ch8('hello ', 50, net, vocab, device)

'he<unk><unk>o<unk>tomakeahumanlineofsucheducationdangersbythesunsetithoughtatthatitwasanightmarethinginthedarkiwasatendencyandiwasthemtoaslightbeachofakindisawthenoisewretchednessformymachineifeltsaw'